# 訓練路考用的 YOLO（Colab 版）

跟自己電腦上的 `train_yolo.py` 是同一支程式，只是在 Colab 的 GPU 上跑，比較快。

開始前：上方選單「執行階段 → 變更執行階段類型」選 **T4 GPU**。

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime onnxslim
!nvidia-smi -L

## 1 下載你們組標好的資料

1. 在 Roboflow 的專案頁按 **Download Dataset**，網址長得像 `app.roboflow.com/<workspace>/<project>/<版本>`，把這三個填到下面。
2. API key 不要直接寫在筆記本裡。點左邊的鑰匙圖示「密鑰」，新增一個名稱 `ROBOFLOW_API_KEY`、值是你的 key，並打開「筆記本存取權」。

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = "你的-workspace"
PROJECT = "你的-project"
VERSION = 1

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location="dataset", overwrite=True)

## 2 訓練

下面這格就是 `train_yolo.py`。只改「設定」那一段，`DATA` 不用改。

In [ ]:
# 訓練路考用的 YOLO
# 自己電腦和 Colab 跑的是同一支程式。只改下面「設定」這一段。
# 跑完會得到一個 groupXX_confY.onnx，就是要上傳的檔案。
from pathlib import Path
import shutil

import yaml
from ultralytics import YOLO

# 設定
GROUP = 1                   # 組別
CONF = 0.35                 # 路考時，信心度到這個值才當成「看見」。寧可多煞車就調低，寧可少誤判就調高
DATA = "dataset/data.yaml"  # Roboflow 匯出（YOLO 格式）解壓後裡面的 data.yaml
MODEL = "yolo26n.pt"        # 從哪個預訓練模型開始：n 最小最快，s、m 越來越大
EPOCHS = 100                # 沒有 NVIDIA 顯卡只能用 CPU，一輪要一分鐘以上；先用 3 確認資料沒問題，正式訓練到 Colab
IMGSZ = 640                 # 訓練時圖片縮放成的大小；號誌很小，調大看得比較清楚，但比較慢
BATCH = -1                  # -1 讓程式自己決定；記憶體不夠時改成 8 或 4

# 標註規範規定的類別名稱，大小寫要一樣，順序不限
CLASSES = {"car", "pedestrian", "bicyclist", "sign"}


def check_classes(data):
    names = yaml.safe_load(Path(data).read_text(encoding="utf-8"))["names"]
    names = set(names.values() if isinstance(names, dict) else names)
    if names != CLASSES:
        raise SystemExit(f"類別名稱跟標註規範不一樣：多了 {names - CLASSES or '無'}，少了 {CLASSES - names or '無'}。"
                         "請回 Roboflow 改類別名稱後重新匯出。")


def train(group, conf, data, model, epochs, imgsz, batch):
    check_classes(data)
    run = YOLO(model)
    run.train(data=data, epochs=epochs, imgsz=imgsz, batch=batch, name=f"group{group:02d}")

    best = YOLO(run.trainer.best)
    print("\n用驗證資料看各類別的表現（mAP50：0 到 1，越高越好）")
    metrics = best.val(data=data, imgsz=imgsz)
    for i, value in enumerate(metrics.box.ap50):
        print(f"  {best.names[metrics.box.ap_class_index[i]]:10s} {value:.2f}")

    upload = Path(f"group{group:02d}_conf{conf:.2f}.onnx")
    shutil.copy(best.export(format="onnx", imgsz=imgsz), upload)
    print(f"\n要上傳的檔案：{upload.resolve()}")
    return upload


if __name__ == "__main__":
    train(GROUP, CONF, DATA, MODEL, EPOCHS, IMGSZ, BATCH)

## 3 下載要上傳的檔案

下載下來的 `groupXX_confY.onnx` 就是要交的檔案，檔名不要改。

In [ ]:
from google.colab import files

files.download(f"group{GROUP:02d}_conf{CONF:.2f}.onnx")